# 03. 위험 점수 계산 (Risk Score)

이 Notebook에서 하는 일:
1. 현금 위험 점수 (Cash Risk) — 버킷 1 충분성
2. 순서 위험 점수 (Sequence Risk) — 하락장 취약성
3. 집중 위험 점수 (Concentration Risk) — 자산 편중
4. 종합 위험 등급 산출 (녹색 / 황색 / 적색)
5. 위험 게이지 시각화
6. DB 저장

In [1]:
# 공통 설정
import pandas as pd
import numpy as np
import yaml, sqlite3, os
from pathlib import Path
from dotenv import load_dotenv
from datetime import date

load_dotenv()
PROJECT_ROOT = Path.cwd()

with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
TARGET          = CONFIG['portfolio']

print(f"사용자: {CONFIG['user']['name']}")
print(f"월 생활비: {MONTHLY_EXPENSE:,}원")

사용자: 종헌
월 생활비: 2,500,000원


In [2]:
# 자산 데이터 로드
assets_path = PROJECT_ROOT / 'assets.csv'
sample_path = PROJECT_ROOT / 'assets_sample.csv'

if assets_path.exists():
    df = pd.read_csv(assets_path, encoding='utf-8-sig')
else:
    df = pd.read_csv(sample_path, encoding='utf-8-sig')
    print('※ 샘플 데이터 사용 중')

mask = df['current_value'].isna() | (df['current_value'] == 0)
df.loc[mask, 'current_value'] = df.loc[mask, 'quantity'] * df.loc[mask, 'unit_price']
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)

BUCKET_MAP = {'cash': 1, 'bond': 2, 'tdf': 2, 'equity': 3, 'income': 3}
df['bucket'] = df['asset_type'].str.lower().map(BUCKET_MAP).fillna(0).astype(int)

bucket_sum = df.groupby('bucket')['current_value'].sum()
total = df['current_value'].sum()
b1 = bucket_sum.get(1, 0)
b2 = bucket_sum.get(2, 0)
b3 = bucket_sum.get(3, 0)

print(f'총 자산: {total:,.0f}원')
print(f'버킷 1 (현금성): {b1:,.0f}원')
print(f'버킷 2 (채권):   {b2:,.0f}원')
print(f'버킷 3 (성장):   {b3:,.0f}원')

※ 샘플 데이터 사용 중
총 자산: 67,550,000원
버킷 1 (현금성): 35,000,000원
버킷 2 (채권):   31,130,000원
버킷 3 (성장):   1,420,000원


## 1. 현금 위험 점수 (Cash Risk)

버킷 1(현금성 자산)이 생활비 몇 개월치인지에 따라 점수를 부여합니다.

| 기준 | 점수 |
|------|------|
| 12개월 이상 | 0점 (안전) |
| 6~12개월 | 30점 (주의) |
| 3~6개월 | 60점 (경고) |
| 3개월 미만 | 100점 (위험) |

In [3]:
months_covered = b1 / MONTHLY_EXPENSE if MONTHLY_EXPENSE > 0 else 0

if months_covered >= 12:
    cash_score = 0
    cash_level = '안전'
    cash_emoji = '🟢'
elif months_covered >= 6:
    cash_score = 30
    cash_level = '주의'
    cash_emoji = '🟡'
elif months_covered >= 3:
    cash_score = 60
    cash_level = '경고'
    cash_emoji = '🟠'
else:
    cash_score = 100
    cash_level = '위험'
    cash_emoji = '🔴'

print(f'=== 현금 위험 점수 ===')
print(f'{cash_emoji} 등급: {cash_level}')
print(f'버킷 1: {b1:,.0f}원 ({months_covered:.1f}개월치)')
print(f'현금 위험 점수: {cash_score}점 / 100점')

=== 현금 위험 점수 ===
🟢 등급: 안전
버킷 1: 35,000,000원 (14.0개월치)
현금 위험 점수: 0점 / 100점


## 2. 순서 위험 점수 (Sequence of Returns Risk)

은퇴 초기 하락장이 발생하면 회복이 어렵습니다.  
주식 비중이 높을수록, 버킷 1이 적을수록 순서 위험이 커집니다.

| 조건 | 점수 |
|------|------|
| 주식 비중 50% 초과 + 버킷1 6개월 미만 | 100점 |
| 주식 비중 35~50% | 50점 |
| 주식 비중 35% 미만 | 20점 |

In [4]:
# 주식형 비중 계산 (equity + income)
type_sum = df.groupby('asset_type')['current_value'].sum()
equity_amt = type_sum.get('equity', 0) + type_sum.get('income', 0)
equity_ratio = equity_amt / total if total > 0 else 0

if equity_ratio > 0.50 and months_covered < 6:
    seq_score = 100
    seq_level = '위험 (주식 비중 과다 + 현금 부족)'
    seq_emoji = '🔴'
elif equity_ratio > 0.50:
    seq_score = 70
    seq_level = '경고 (주식 비중 과다)'
    seq_emoji = '🟠'
elif equity_ratio > 0.35:
    seq_score = 50
    seq_level = '주의'
    seq_emoji = '🟡'
else:
    seq_score = 20
    seq_level = '안전'
    seq_emoji = '🟢'

print(f'=== 순서 위험 점수 ===')
print(f'{seq_emoji} 등급: {seq_level}')
print(f'주식형 자산: {equity_amt:,.0f}원 (비중 {equity_ratio*100:.1f}%)')
print(f'순서 위험 점수: {seq_score}점 / 100점')

=== 순서 위험 점수 ===
🟢 등급: 안전
주식형 자산: 1,420,000원 (비중 2.1%)
순서 위험 점수: 20점 / 100점


## 3. 집중 위험 점수 (Concentration Risk)

특정 자산군에 지나치게 집중된 경우 위험이 높아집니다.  
목표 비중 대비 실제 비중의 최대 이탈폭으로 측정합니다.

In [5]:
current = {
    'cash':   type_sum.get('cash', 0) / total if total > 0 else 0,
    'bond':   (type_sum.get('bond', 0) + type_sum.get('tdf', 0)) / total if total > 0 else 0,
    'equity': type_sum.get('equity', 0) / total if total > 0 else 0,
    'income': type_sum.get('income', 0) / total if total > 0 else 0,
}

targets = {
    'cash':   TARGET['target_cash'],
    'bond':   TARGET['target_bond'],
    'equity': TARGET['target_equity'],
    'income': TARGET['target_income'],
}

max_deviation = max(abs(current[k] - targets[k]) for k in targets)

if max_deviation >= 0.20:
    conc_score = 100
    conc_level = '위험 (20%p 이상 이탈)'
    conc_emoji = '🔴'
elif max_deviation >= 0.10:
    conc_score = 50
    conc_level = '주의 (10~20%p 이탈)'
    conc_emoji = '🟡'
else:
    conc_score = 0
    conc_level = '안전 (10%p 미만)'
    conc_emoji = '🟢'

print(f'=== 집중 위험 점수 ===')
print(f'{conc_emoji} 등급: {conc_level}')
print(f'목표 대비 최대 이탈: {max_deviation*100:.1f}%p')
print()
labels_map = {'cash': '현금성', 'bond': '채권/TDF', 'equity': '주식형', 'income': '리츠/인컴'}
for k, label in labels_map.items():
    diff = current[k] - targets[k]
    flag = ' ← ⚠' if abs(diff) >= 0.10 else ''
    print(f'  {label:8s}: 목표 {targets[k]*100:.0f}%  현재 {current[k]*100:.1f}%  차이 {diff*100:+.1f}%p{flag}')
print(f'집중 위험 점수: {conc_score}점 / 100점')

=== 집중 위험 점수 ===
🔴 등급: 위험 (20%p 이상 이탈)
목표 대비 최대 이탈: 33.5%p

  현금성     : 목표 25%  현재 51.8%  차이 +26.8%p ← ⚠
  채권/TDF  : 목표 25%  현재 46.1%  차이 +21.1%p ← ⚠
  주식형     : 목표 35%  현재 1.5%  차이 -33.5%p ← ⚠
  리츠/인컴   : 목표 15%  현재 0.6%  차이 -14.4%p ← ⚠
집중 위험 점수: 100점 / 100점


## 4. 종합 위험 점수 및 등급

In [6]:
# 가중 평균 (현금 40%, 순서 40%, 집중 20%)
total_score = cash_score * 0.40 + seq_score * 0.40 + conc_score * 0.20

if total_score <= 25:
    risk_level = 'green'
    risk_label = '녹색 (안전)'
    risk_emoji = '🟢'
    risk_action = '현재 포트폴리오를 유지하세요.'
elif total_score <= 55:
    risk_level = 'yellow'
    risk_label = '황색 (주의)'
    risk_emoji = '🟡'
    risk_action = '1~2개월 내 포트폴리오 점검을 권장합니다.'
else:
    risk_level = 'red'
    risk_label = '적색 (위험)'
    risk_emoji = '🔴'
    risk_action = '즉시 리밸런싱 또는 버킷 충전이 필요합니다.'

print('=' * 50)
print('  종합 위험 점수')
print('=' * 50)
print()
print(f'  현금 위험 점수:   {cash_score:>5.0f}점  (가중치 40%)')
print(f'  순서 위험 점수:   {seq_score:>5.0f}점  (가중치 40%)')
print(f'  집중 위험 점수:   {conc_score:>5.0f}점  (가중치 20%)')
print(f'  {'─'*35}')
print(f'  종합 위험 점수:   {total_score:>5.1f}점 / 100점')
print()
print(f'  {risk_emoji} 위험 등급: {risk_label}')
print(f'  → {risk_action}')

  종합 위험 점수

  현금 위험 점수:       0점  (가중치 40%)
  순서 위험 점수:      20점  (가중치 40%)
  집중 위험 점수:     100점  (가중치 20%)
  ───────────────────────────────────
  종합 위험 점수:    28.0점 / 100점

  🟡 위험 등급: 황색 (주의)
  → 1~2개월 내 포트폴리오 점검을 권장합니다.


## 5. 위험 게이지 시각화

In [7]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = 'iframe'

# 게이지 색상
needle_color = {'green': '#00B050', 'yellow': '#FFC000', 'red': '#FF0000'}[risk_level]

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'bar'}]],
    subplot_titles=['종합 위험 게이지', '항목별 위험 점수']
)

# 게이지 차트
fig.add_trace(go.Indicator(
    mode='gauge+number',
    value=total_score,
    title={'text': f'{risk_emoji} {risk_label}', 'font': {'size': 16}},
    gauge={
        'axis': {'range': [0, 100], 'tickwidth': 1},
        'bar': {'color': needle_color},
        'steps': [
            {'range': [0, 25],  'color': '#E2EFDA'},
            {'range': [25, 55], 'color': '#FFF2CC'},
            {'range': [55, 100],'color': '#FCE4D6'},
        ],
        'threshold': {
            'line': {'color': 'black', 'width': 3},
            'thickness': 0.75,
            'value': total_score
        }
    }
), row=1, col=1)

# 항목별 막대 차트
categories = ['현금 위험', '순서 위험', '집중 위험']
scores     = [cash_score, seq_score, conc_score]
bar_colors = []
for s in scores:
    if s <= 25:
        bar_colors.append('#00B050')
    elif s <= 55:
        bar_colors.append('#FFC000')
    else:
        bar_colors.append('#FF0000')

fig.add_trace(go.Bar(
    x=categories,
    y=scores,
    marker_color=bar_colors,
    text=[f'{s:.0f}점' for s in scores],
    textposition='outside'
), row=1, col=2)

fig.update_layout(
    title_text=f'포트폴리오 위험 분석  |  {date.today().strftime("%Y년 %m월 %d일")}',
    title_font_size=16,
    height=400,
    font=dict(size=13),
    showlegend=False
)
fig.update_yaxes(range=[0, 110], row=1, col=2)
fig.show()

## 6. 위험 점수 DB 저장

In [8]:
today   = str(date.today())
db_path = PROJECT_ROOT / 'portfolio.db'

conn = sqlite3.connect(db_path)
cur  = conn.cursor()

cur.execute('SELECT id FROM risk_scores WHERE date = ?', (today,))
row = cur.fetchone()

if row:
    cur.execute('''
        UPDATE risk_scores
        SET total_score=?, cash_score=?, seq_score=?, conc_score=?, level=?
        WHERE date=?
    ''', (total_score, cash_score, seq_score, conc_score, risk_level, today))
    print(f'오늘({today}) 위험 점수 업데이트 완료')
else:
    cur.execute('''
        INSERT INTO risk_scores (date, total_score, cash_score, seq_score, conc_score, level)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (today, total_score, cash_score, seq_score, conc_score, risk_level))
    print(f'오늘({today}) 위험 점수 저장 완료')

conn.commit()
conn.close()

print()
print(f'  종합 점수: {total_score:.1f}점  등급: {risk_label}')

오늘(2026-05-20) 위험 점수 저장 완료

  종합 점수: 28.0점  등급: 황색 (주의)


## 7. 위험 점수 이력 조회

In [9]:
conn = sqlite3.connect(db_path)
history = pd.read_sql_query(
    'SELECT date, total_score, cash_score, seq_score, conc_score, level FROM risk_scores ORDER BY date DESC LIMIT 12',
    conn
)
conn.close()

if history.empty:
    print('위험 점수 이력이 없습니다.')
else:
    level_map = {'green': '🟢 녹색', 'yellow': '🟡 황색', 'red': '🔴 적색'}
    history['level'] = history['level'].map(level_map).fillna(history['level'])
    history.columns = ['날짜', '종합', '현금', '순서', '집중', '등급']
    print('=== 최근 위험 점수 이력 ===')
    print(history.to_string(index=False))

=== 최근 위험 점수 이력 ===
        날짜   종합  현금   순서    집중   등급
2026-05-20 28.0 0.0 20.0 100.0 🟡 황색


---

## 완료!

- 현금·순서·집중 위험 점수가 계산되었습니다.
- 게이지 차트로 현재 위험 수준을 한눈에 확인하세요.

**다음 단계**: `04_rebalance.ipynb` — 리밸런싱 조정안 계산